# 服务器 GPU 运行 —— full-panel BC(proactive Rail Route)+ 评估 + 基线阶梯

**目的(D57/D58)**:在**新 proactive narrow 环境**上克隆信号员**完整面板决策** `{WAIT/SET/CANCEL}`(时机+选路+入区放行),
取代旧"只克隆选路"的 BC;然后闭环评估(RR 口径:完成率/加权次生延误/PPM 对计划/停车)并与零学习基线阶梯对比。

**纪律(务必)**:
- 信号员是**参照基线非靶子**(D58);拟合型模型看 **held-out**(D40/D43):BC/lookup 只在 val 选型、**test 全程不碰**。
- 环境 **narrow**(人类锚,D57.1);`reward_mode=per_train`(D58);PPM=**对 CIF 计划的绝对准点**(非 secondary、非以信号员为标准)。
- **D45 作废**:先直接试 RL、看 EV 是否自恢复,**再决定**要不要按天差分/half-life 补丁,不预设。
- 全量在服务器跑;`.ipynb` 是唯一允许出现服务器路径的地方;脚本一律 `PYTHONPATH=.` 相对路径。

## 设置(你的服务器前置)

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project

PROJ = '/rds/homes/h/hxn886/ondemand/final_project'
PY = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX = (f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
          f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK. shell cell 用  !{PREFIX} <script> <args>')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1) 全量 full-panel BC 数据集(96 运营日)
narrow proactive 回放逐 epoch 记 `(obs, options, chosen_idx, signal)`;D34 划分(train=9–10月 / holdout=11–12月);
`combine` 出 `bc_fullpanel_all.parquet` + 指纹 + 分层报告。(幂等:已存在的逐日文件跳过。)

In [ ]:
!{PREFIX} scripts/30_build_bc_fullpanel_dataset.py gen all

In [ ]:
!{PREFIX} scripts/30_build_bc_fullpanel_dataset.py combine

## 2) 训 full-panel BC(克隆 `chosen_idx` = 整个面板动作)
`PanelOptionPolicy`(set 支=M11 PointerBC,整体可热启动 RL);对 `chosen_idx` 做 NLL;`route_prior` 只从 train 段拟合。
**四编码器消融(D9)**:deepsets(主)/ mlp / settransformer / gnn。产物 `artifacts/dataset/bc_fullpanel_<enc>.pt`。

In [ ]:
!{PREFIX} scripts/31_train_bc_fullpanel.py deepsets 30
!{PREFIX} scripts/31_train_bc_fullpanel.py mlp 30
!{PREFIX} scripts/31_train_bc_fullpanel.py settransformer 30
!{PREFIX} scripts/31_train_bc_fullpanel.py gnn 30

## 3) 闭环评估(RR 口径) + 基线阶梯
**headline**:完成率 · 加权次生延误(λ) · PPM(对计划,搁浅=不准点) · 非计划停车数;按 λ 分层。
**看 val(干净 held-out);test 最终一次性 headline 才碰。** BC 闭环 vs 信号员参照 vs 零学习基线。

In [ ]:
# 基线阶梯(零学习): 打不过这些的学习方法没有价值(D26)。lookup/fcfs/fsfs/greedy/random。
!{PREFIX} scripts/32_baseline_ladder.py val

In [ ]:
# 三方(实为 6 方)闭环对比: **4 个 BC(每编码器一行)** vs 信号员参照 vs greedy(同 RR 口径, val)。
# --bc 支持 glob → 一次评估全部编码器(D9 消融, 论文结果更丰富); 延误已统一到地平线(D46)。
!{PREFIX} scripts/29_eval_metric_suite.py val --bc 'artifacts/dataset/bc_fullpanel_*.pt'
# 最终 headline 才在 test 上跑一次: val→test

## 4) 下一步(RL,单独 notebook)
- full-panel BC 作**热启动 + KL 锚**(D44)训 maskable PPO(`reward_mode=per_train`,D58)。
- **先直接试**:看 EV 是否自恢复(D45 作废、不预设需要按天差分/half-life)。
- **wide 窗口作 RL 消融**(=RR 自由:能否更早预留、超越信号员的时机)。
- Step9(选路偏离诊断)在新环境重做。